In [ ]:
import os
import pandas as pd

# Directory containing all correlation matrices
corr_dir = 'correlation_matrices_all_subjects'

# List all CSV files in the directory
corr_files = [f for f in os.listdir(corr_dir) if f.endswith('.csv')]

# Dictionary to store matrices: key = filename, value = DataFrame
correlation_matrices = {}
for fname in corr_files:
    fpath = os.path.join(corr_dir, fname)
    correlation_matrices[fname] = pd.read_csv(fpath, index_col=0)

# Example: print the keys and shape of one matrix
print(f"Loaded {len(correlation_matrices)} correlation matrices.")

Loaded 129 correlation matrices.


In [ ]:
# community detection: Spectral Clustering (number of communities can be set)
from sklearn.cluster import SpectralClustering
import numpy as np

# Set desired number of communities (e.g., 3 or 4)
n_communities = 4

spectral_community_assignments = {}

for fname, matrix in correlation_matrices.items():
    # Convert to absolute value and fill diagonal with 1 (self-correlation)
    mat = matrix.abs().values
    np.fill_diagonal(mat, 1)
    # Spectral clustering expects a similarity matrix
    clustering = SpectralClustering(n_clusters=n_communities, affinity='precomputed', assign_labels='kmeans', random_state=0)
    labels = clustering.fit_predict(mat)
    # Map channel names to community labels
    node_to_comm = dict(zip(matrix.index, labels))
    spectral_community_assignments[fname] = node_to_comm

# Example: print Spectral Clustering community assignment for one subject/session
first_key = next(iter(spectral_community_assignments))
print(f"Spectral Clustering community assignment for {first_key}:")
print(spectral_community_assignments[first_key])

Spectral Clustering community assignment for sub-NORB00064_ses-2_correlation_matrix_avg.csv:
{'Fp1': np.int32(0), 'Fp2': np.int32(0), 'F3': np.int32(0), 'F4': np.int32(0), 'C3': np.int32(3), 'C4': np.int32(3), 'P3': np.int32(2), 'P4': np.int32(1), 'O1': np.int32(2), 'O2': np.int32(1), 'F7': np.int32(0), 'F8': np.int32(0), 'T3': np.int32(2), 'T4': np.int32(1), 'T5': np.int32(2), 'T6': np.int32(1), 'FZ': np.int32(0), 'CZ': np.int32(3), 'PZ': np.int32(3)}


In [10]:
# Calculate node degree (sum of connection strengths) for each node in each matrix
node_degrees = {}

for fname, matrix in correlation_matrices.items():
    # Use absolute value of the matrix for degree calculation
    abs_matrix = matrix.abs()
    # Degree: sum of weights for each node (row)
    degrees = abs_matrix.sum(axis=1)
    node_degrees[fname] = degrees

# Example: print node degrees for one subject/session
first_key = next(iter(node_degrees))
print(f"Node degrees for {first_key}:")
print(node_degrees[first_key])

Node degrees for sub-NORB00064_ses-2_correlation_matrix_avg.csv:
Fp1     9.672434
Fp2     9.761875
F3      9.917231
F4     10.520209
C3      9.758827
C4      9.804729
P3      9.189135
P4      9.324550
O1      6.460069
O2      7.077024
F7      8.769347
F8      9.747924
T3      6.998727
T4      8.711332
T5      6.593701
T6      7.150652
FZ      9.721601
CZ      8.256914
PZ      9.637366
dtype: float64


In [11]:
# Calculate participation coefficient for each node in each matrix using Spectral Clustering communities
participation_coefficients = {}

for fname, matrix in correlation_matrices.items():
    communities = spectral_community_assignments[fname]
    degrees = node_degrees[fname]
    abs_matrix = matrix.abs()
    # Get unique community labels
    unique_comms = set(communities.values())
    pc = {}
    for node in abs_matrix.index:
        k_i = degrees[node]
        if k_i == 0:
            pc[node] = 0.0
            continue
        sum_sq = 0.0
        for comm in unique_comms:
            # Nodes in this community
            comm_nodes = [n for n, c in communities.items() if c == comm]
            # Sum of weights from node to nodes in this community
            k_im = abs_matrix.loc[node, comm_nodes].sum()
            sum_sq += (k_im / k_i) ** 2
        pc[node] = 1 - sum_sq
    participation_coefficients[fname] = pc

# Example: print participation coefficients for one subject/session
first_key = next(iter(participation_coefficients))
print(f"Participation coefficients for {first_key}:")
print(participation_coefficients[first_key])

Participation coefficients for sub-NORB00064_ses-2_correlation_matrix_avg.csv:
{'Fp1': np.float64(0.5869047662185287), 'Fp2': np.float64(0.6034675061708927), 'F3': np.float64(0.5834410868995324), 'F4': np.float64(0.6177020269428602), 'C3': np.float64(0.7043629930753252), 'C4': np.float64(0.7032451542595712), 'P3': np.float64(0.7438028440083202), 'P4': np.float64(0.7380775707854936), 'O1': np.float64(0.7074544089970367), 'O2': np.float64(0.727190383594958), 'F7': np.float64(0.6286930795576743), 'F8': np.float64(0.6273214903094838), 'T3': np.float64(0.7079124749438707), 'T4': np.float64(0.7130543327045942), 'T5': np.float64(0.6962385924056423), 'T6': np.float64(0.7086818485134866), 'FZ': np.float64(0.5984797799414848), 'CZ': np.float64(0.6909332578776726), 'PZ': np.float64(0.7473580239774146)}
